# Ablation run — single-pass baseline vs full pipeline (spec §8 / M5)

Purpose-built for one job, so it can be run top to bottom without touching the
VLM sections in `flowmind_colab.ipynb`.

**Before running:** attach a **GPU** runtime. In VS Code that's
`Select Kernel → Colab → GPU`; in the browser, `Runtime → Change runtime type`.

**Runs on `p/examiner`, not `main`** — the Examiner only exists on that branch.

Budget about an hour. Roughly 30 GB of model weights come down (Qwen3-8B for
answering, Mistral-7B for judging) plus generation time. Model weights go to the
VM disk, not Drive: Qwen3-8B alone is 16 GB and free Drive is 15 GB total. That
means a recycled runtime re-downloads them, which is the accepted trade.

Results are written to `runs/` which is symlinked to Drive, so output survives a
disconnect even if the weights don't.

## 0. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU - change the runtime type before continuing'

## 1. Setup

Idempotent — safe to re-run on a fresh runtime. Expect the log line to show
branch `p/examiner` and **52 passed**. If it shows `main`, stop: the Examiner
isn't there.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/FlowMind')
REPO = Path('/content/FlowMind')

if not REPO.exists():
    !git clone -q https://github.com/PurvajaNarayan/FlowMind.git {REPO}
!cd {REPO} && git fetch -q origin && git checkout -q p/examiner && git pull -q --ff-only
!cd {REPO} && git log --oneline -1 && git branch --show-current

# Point the gitignored paths at Drive. `runs` matters most here: it is where the
# generated answers land, and re-generating them is the expensive part.
for rel in ('data/train_full.json', 'data/test_full.json', 'data/images',
            'models', 'runs'):
    link, target = REPO / rel, DRIVE / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    elif link.exists():
        link.unlink()
    link.symlink_to(target)
    print(f'{rel:24} -> {target}')

# bitsandbytes is NOT in requirements-vlm.txt (commented out there). Without it,
# 4-bit loading fails and Qwen3-8B in fp16 (~16GB) will not fit a 15GB T4.
!cd {REPO} && pip install -q -r requirements.txt && pip install -q bitsandbytes accelerate
!cd {REPO} && python -m pytest -q 2>&1 | tail -2

Mounted at /content/drive
34b9a80 (HEAD -> p/examiner, origin/p/examiner) Add a purpose-built notebook for the ablation run
p/examiner
data/train_full.json     -> /content/drive/MyDrive/FlowMind/data/train_full.json
data/test_full.json      -> /content/drive/MyDrive/FlowMind/data/test_full.json
data/images              -> /content/drive/MyDrive/FlowMind/data/images
models                   -> /content/drive/MyDrive/FlowMind/models
runs                     -> /content/drive/MyDrive/FlowMind/runs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 56.7 MB/s eta 0:00:00:00:0100:01
....................................................                     [100%]
52 passed in 4.12s


## 2. Smoke test — 12 items

Most of the time here is the 16 GB Qwen3-8B download, not generation. Three
things to check in the output:

- **no `[llm] 4-bit unavailable`** — if that appears, bitsandbytes didn't take and
  the model won't fit
- topological rows show `OK`/`X`; content rows show `--`, meaning recorded but not
  scored inline (correct — the judge does those in step 4)
- `examiner fired a revision : N` — any value; it just proves the loop runs

In [3]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/run_ablation.py --n 12 --save runs/abl_smoke.jsonl

backend LocalTransformersClient | model Qwen/Qwen3-8B
12 items over 12 charts

config.json: 100% 728/728 [00:00<00:00, 3.60MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 29.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 50.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 107MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 8.93MB/s,   ???B/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 30.0MB/s,   ???B/s  ]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 79.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.96G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/11.9G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   

## 3. Full run — 60 items

Weights are cached from step 2, so this is generation only. 60 items spread over
the 12 (subset × question-type) cells, max 2 questions per flowchart.

In [4]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/run_ablation.py --n 60 --save runs/ablation_v1.jsonl

backend LocalTransformersClient | model Qwen/Qwen3-8B
60 items over 57 charts

Loading weights: 100% 399/399 [00:04<00:00, 88.02it/s] 
code00093        [topological     ] baseline=OK pipeline=OK (graph_tool, rev=0)
code00148        [fact_retrieval  ] baseline=-- pipeline=-- (examiner, rev=0)
code00511        [applied_scenario] baseline=-- pipeline=-- (examiner, rev=0)
code00011        [flow_referential] baseline=-- pipeline=-- (examiner, rev=0)
instruct00494    [topological     ] baseline=OK pipeline=OK (graph_tool, rev=0)
instruct00522    [fact_retrieval  ] baseline=-- pipeline=-- (examiner, rev=0)
instruct00162    [applied_scenario] baseline=-- pipeline=-- (examiner, rev=0)
instruct00520    [flow_referential] baseline=-- pipeline=-- (examiner, rev=0)
wiki00057        [topological     ] baseline=OK pipeline=OK (graph_tool, rev=0)
wiki00701        [fact_retrieval  ] baseline=-- pipeline=-- (examiner, rev=0)
wiki00222        [applied_scenario] baseline=-- pipeline=-- (examiner, rev=0)
w

## 4. Score both arms with the judge

Downloads Mistral-7B (~15 GB) once. Read the output in this order:

1. **`judge agrees with exact match: N/…`** — the judge measured against known
   truth on the topological questions. If this is low, treat everything below as
   correspondingly noisy.
2. **content accuracy by arm, and the delta** — the project's headline, and the
   first real number for the 57% of the benchmark that needs language.
3. **`unparsed judge replies`** — these are excluded from both numerator and
   denominator, so a large count means the judge is struggling with the format
   rather than that answers were wrong.

In [5]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/score_run.py runs/ablation_v1.jsonl --save runs/ablation_v1_scored.jsonl

judge model: mistralai/Mistral-7B-Instruct-v0.3
config.json: 100% 601/601 [00:00<00:00, 2.97MB/s]
tokenizer_config.json: 100% 141k/141k [00:00<00:00, 147MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 48.1MB/s]

tokenizer.model: downloading bytes:  14% 81.9k/587k [00:00<00:03, 146kB/s]
tokenizer.model: downloading bytes: 100% 448k/448k [00:00<00:00, 714kB/s, 44.3kB/s  ]
tokenizer.model: reconstructing file: 100% 587k/587k [00:00<00:00, 936kB/s, 58.1kB/s  ]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.19MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 75.4MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.95G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/9.95G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  19% 2.76G/14.5G [00:09<01:23, 140MB/s,  120MB/s  ]
Reconstructing (incomplete total...): 

### If step 4 fails on a gated repo

`mistralai/*` repos sometimes require accepting terms. Either accept on the HF
model page and set `HF_TOKEN`, or fall back to Phi-4-mini, which is MIT and
ungated — the cell below does that. Note in the write-up which judge produced the
numbers.

In [6]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
# only needed if the cell above failed to download the judge
!cd /content/FlowMind && HF_HOME=/content/hf_llm FLOWMIND_JUDGE_MODEL=microsoft/Phi-4-mini-instruct python tools/score_run.py runs/ablation_v1.jsonl --save runs/ablation_v1_scored.jsonl

judge model: microsoft/Phi-4-mini-instruct
config.json: 100% 2.50k/2.50k [00:00<00:00, 7.13MB/s]
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
tokenizer_config.json: 100% 2.93k/2.93k [00:00<00:00, 4.53MB/s]

tokenizer.json: downloading bytes:  27% 4.21M/15.5M [00:00<00:01, 7.09MB/s]
tokenizer.json: downloading bytes: 100% 5.00M/5.00M [00:00<00:00, 8.14MB/s,  495kB/s  ]
tokenizer.json: reconstructing file: 100% 15.5M/15.5M [00:00<00:00, 25.2MB/s, 1.54MB/s  ]
added_tokens.json: 100% 249/249 [00:00<00:00, 1.21MB/s]
special_tokens_map.json: 100% 587/587 [00:00<00:00, 349kB/s]
model.safetensors.index.json: 100% 16.3k/16.3k [00:00<00:00, 35.4MB/s]
Recons

---
## 5. Is the judge actually a measurement? (negative controls)

Run this **before** believing any content number. The first ablation scored
the same 90 answers 20 points apart under two judges, and the lenient one
returned 100.0% for a single 8B pass while saying "incorrect" twice in 90
rows — it marked both `'7 steps.'` and `'6 steps.'` correct for one question.

Step 4's topological agreement check missed that entirely, because judging
`9` and `Yes` is trivial. This probes both directions: does the judge accept
real answers, and does it **reject** answers that are definitely wrong.

**Read TNR first.** Below 50% and the content numbers are not reportable, no
matter how good they look.

In [ ]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/validate_judge.py --n 25 --save runs/judge_validation_mistral.jsonl

Same probe against the other judge, for comparison:

In [ ]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm FLOWMIND_JUDGE_MODEL=microsoft/Phi-4-mini-instruct \
  python tools/validate_judge.py --n 25 --save runs/judge_validation_phi.jsonl

---
## 6. Representation test: is the pipeline worse, or is the serialization worse?

The pipeline came in below the baseline on content (−4.4 / −8.9 points). The
revision loop cannot explain it — it fired twice in 60 items, so the content
arm was effectively a single call.

What did differ is the input: the Examiner reads the node/edge listing, the
baseline reads raw Mermaid. `--representation mermaid` makes the Examiner read
the same thing the baseline does, with an identical system prompt, so the only
remaining difference is the loop.

- **deficit disappears** → it was a formatting choice, not the architecture
- **deficit persists** → it really is the pipeline, which is a finding

Weights are cached by now, so each of these is generation only.

In [ ]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/run_ablation.py --n 60 \
  --representation mermaid --save runs/ablation_mermaid.jsonl

In [ ]:
!cd /content/FlowMind && git pull -q --ff-only && git log --oneline -1
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/score_run.py \
  runs/ablation_mermaid.jsonl --save runs/ablation_mermaid_scored.jsonl

Compare the `examiner` row against `runs/ablation_v1_scored.jsonl` from step 4.
Same items, same seed, same judge — the only change is what the Examiner was
shown.